# Vie-GameEmo — Training

**Notebook này thực hiện:**
1. Load annotations từ Stage 0
2. Trích xuất và cache features (AST, ViT-FER, ViT-ImageNet, XLM-R)
3. **Stage 1 — Perception**: huấn luyện Fusion + Classifier (30 epochs)
4. **Stage 2 — Cognition**: joint training LLM + adapter (tùy chọn)
5. Eval nhanh trên val split
6. Lưu checkpoint

**Yêu cầu Kaggle:**
- Accelerator: **GPU T4 x1** (hoặc P100)
- Internet: **BẬT** (để tải model weights từ HuggingFace)
- Input: Dataset chứa annotations từ Stage 0
- Runtime: ~2-4 giờ (feature extraction + 30 epochs training)

**Cách thêm annotations:**  
1. Upload file `stage0_annotations.zip` lên Kaggle Dataset  
2. Thêm vào notebook input: *Add data → Your Datasets*

In [ ]:
# ============================================================
# CELL 1 — Môi trường + GPU
# ============================================================
import os, sys, gc, json, shutil
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')
WORKING = '/kaggle/working' if IS_KAGGLE else '/tmp/vie-gameemo'

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | {gpu.total_memory / 1e9:.1f} GB VRAM')
else:
    print('⚠️  Không có GPU')
print(f'Device: {device}')

In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH
# ============================================================

# --- Đường dẫn annotations ---
# Option A: từ Kaggle input dataset (khuyến nghị)
ANNOT_INPUT = '/kaggle/input/vie-gameemo-annotations/annotations'
# Option B: nếu đã upload trực tiếp vào working
ANNOT_LOCAL = os.path.join(WORKING, 'data/annotations')

# Tự động chọn
if os.path.exists(ANNOT_INPUT) and os.listdir(ANNOT_INPUT):
    ANNOT_DIR = ANNOT_INPUT
    print(f'Annotations: Kaggle input → {ANNOT_DIR}')
else:
    ANNOT_DIR = ANNOT_LOCAL
    print(f'Annotations: local → {ANNOT_DIR}')
    print('⚠️  Nếu chưa có annotations, chạy notebook Stage 0 trước')

# --- Đường dẫn project ---
PROJECT_INPUT = '/kaggle/input/vie-gameemo-code'

# --- Hyperparameters ---
FUSION_TYPE     = 'conv_attention_4m'  # 'late' | 'early' | 'mult' | 'conv_attention_4m'
EPOCHS_PERC     = 30     # Perception epochs
BATCH_SIZE      = 16
LR_FUSION       = 2e-4
GRAD_ACCUM      = 4      # effective batch = BATCH_SIZE * GRAD_ACCUM
MIXED_PREC      = 'bf16' # 'bf16' (T4 OK) | 'fp16' | 'no'
EARLY_STOP_PAT  = 5      # patience

# --- Cognition (Stage 2, LLM) ---
TRAIN_COGNITION = False   # True nếu muốn train joint LLM
LLM_MODEL       = 'Qwen/Qwen2.5-7B-Instruct'
EPOCHS_COG      = 5

# --- RLVR (LLM-4) ---
TRAIN_RLVR      = False   # Nặng, cần nhiều VRAM và thời gian

# --- Đường dẫn working ---
DATA_DIR     = os.path.join(WORKING, 'data')
FEAT_DIR     = os.path.join(DATA_DIR, 'features')
CKPT_DIR     = os.path.join(WORKING, 'checkpoints')
LOG_DIR      = os.path.join(WORKING, 'logs')
for d in [FEAT_DIR, CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('\nConfig:')
print(f'  FUSION_TYPE   : {FUSION_TYPE}')
print(f'  EPOCHS_PERC   : {EPOCHS_PERC}')
print(f'  TRAIN_COGNITION: {TRAIN_COGNITION}')
print(f'  TRAIN_RLVR    : {TRAIN_RLVR}')

In [ ]:
# ============================================================
# CELL 3 — Cài thư viện
# ============================================================
!pip install -q \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    peft>=0.13.0 \
    bitsandbytes>=0.43.0 \
    faster-whisper>=1.0.3 \
    librosa>=0.10.1 \
    mediapipe>=0.10.14 \
    pydantic>=2.8.0 \
    pyyaml>=6.0.2 \
    scikit-learn>=1.5.0

if TRAIN_RLVR:
    !pip install -q trl>=0.11.0

print('Done')

In [ ]:
# ============================================================
# CELL 4 — Setup project
# ============================================================
import subprocess

PROJECT_DIR = os.path.join(WORKING, 'vie-gameemo-skeleton')

if os.path.exists(PROJECT_INPUT):
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(PROJECT_INPUT, PROJECT_DIR)
    print(f'Project: Kaggle input → {PROJECT_DIR}')
elif not os.path.exists(PROJECT_DIR):
    GITHUB_URL = 'https://github.com/YOUR_USERNAME/vie-gameemo-skeleton.git'
    subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, PROJECT_DIR], check=True)
    print(f'Project cloned → {PROJECT_DIR}')
else:
    print(f'Project exists: {PROJECT_DIR}')

SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Tạo config
CONFIG_PATH = os.path.join(WORKING, 'config.yaml')
CONFIG = f"""
seed: 42
logging:
  level: INFO
  file: {LOG_DIR}/training.log
  console: true
paths:
  data_root: {DATA_DIR}
  annotations: {ANNOT_DIR}
  features: {FEAT_DIR}
  checkpoints: {CKPT_DIR}
  results: {WORKING}/results
fusion:
  type: {FUSION_TYPE}
  d_model: 768
  n_modalities: 4
  n_conv_blocks: 4
  kernel_size: 3
  align_to: audio
  return_attention: false
classifier:
  hidden_dim: 256
  n_classes: 9
  dropout: 0.3
  loss:
    focal:
      gamma: 2.0
      alpha: 1.0
training:
  perception:
    epochs: {EPOCHS_PERC}
    batch_size: {BATCH_SIZE}
    gradient_accumulation: {GRAD_ACCUM}
    learning_rate:
      fusion: {LR_FUSION}
      classifier: {LR_FUSION}
      encoders: 0.0
    weight_decay: 0.01
    warmup_ratio: 0.1
    grad_clip: 1.0
    mixed_precision: {MIXED_PREC}
    early_stopping:
      enabled: true
      patience: {EARLY_STOP_PAT}
      monitor: val_macro_f1
      mode: max
  cognition:
    epochs: {EPOCHS_COG}
    batch_size: 4
    gradient_accumulation: 8
    learning_rate:
      llm: 2.0e-5
      llm_adapter: 2.0e-4
      fusion: 0.0
      classifier: 0.0
    loss_weights:
      classification: 1.0
      reasoning_lm: 0.5
    lora:
      enabled: true
      rank: 16
      alpha: 32
      target_modules: [q_proj, v_proj]
llm:
  active_setup: llm1
  base_model:
    name: {LLM_MODEL}
    quantization: 4bit
    max_new_tokens: 300
    temperature: 0.7
compute:
  profile: colab
  num_workers: 2
"""
with open(CONFIG_PATH, 'w') as f:
    f.write(CONFIG)
print(f'Config: {CONFIG_PATH}')

In [ ]:
# ============================================================
# CELL 5 — Load annotations + tạo splits
# ============================================================
import random

# Đọc tất cả annotation JSON
annot_files = sorted(Path(ANNOT_DIR).glob('*.json'))
print(f'Tìm thấy {len(annot_files)} annotation files')

if len(annot_files) == 0:
    raise RuntimeError(f'Không có annotation files trong {ANNOT_DIR}. Chạy notebook Stage 0 trước.')

# Đọc và kiểm tra
annotations = []
for p in annot_files:
    try:
        data = json.loads(p.read_text(encoding='utf-8'))
        annotations.append(data)
    except Exception as e:
        print(f'  ⚠️  {p.name}: {e}')

from collections import Counter
label_counts = Counter(a.get('emotion_label', 'unknown') for a in annotations)
print(f'\nTổng hợp lệ: {len(annotations)}')
print('Phân phối nhãn:')
for label, count in sorted(label_counts.items()):
    print(f'  {label:<15}: {count}')

# Tạo splits (70/15/10/5)
random.seed(42)
shuffled = annotations.copy()
random.shuffle(shuffled)
n = len(shuffled)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = int(0.10 * n)

split_assign = {}
for i, a in enumerate(shuffled):
    if i < n_train:
        split_assign[a['clip_id']] = 'train'
    elif i < n_train + n_val:
        split_assign[a['clip_id']] = 'val'
    elif i < n_train + n_val + n_test:
        split_assign[a['clip_id']] = 'test_id'
    else:
        split_assign[a['clip_id']] = 'test_ood'

# Ghi split vào annotation files (trong working dir)
ANNOT_WORK = os.path.join(DATA_DIR, 'annotations')
os.makedirs(ANNOT_WORK, exist_ok=True)
for a in annotations:
    a['split'] = split_assign.get(a['clip_id'], 'train')
    out = os.path.join(ANNOT_WORK, f"{a['clip_id']}.json")
    if not os.path.exists(out) or ANNOT_WORK != ANNOT_DIR:
        with open(out, 'w', encoding='utf-8') as f:
            json.dump(a, f, ensure_ascii=False, indent=2)

# Cập nhật config để trỏ đến working annotations
ANNOT_DIR = ANNOT_WORK

split_counts = Counter(split_assign.values())
print(f'\nSplits: train={split_counts["train"]} val={split_counts["val"]} test_id={split_counts["test_id"]} test_ood={split_counts["test_ood"]}')

## Bước 1 — Trích xuất Features (cache)

In [ ]:
# ============================================================
# CELL 6 — Trích xuất AST Audio Features
# ============================================================
from vie_gameemo.encoders.audio_ast import ASTAudioEncoder
import torch

print('Loading AST audio encoder...')
audio_enc = ASTAudioEncoder(
    model_name='MIT/ast-finetuned-audioset-10-10-0.4593',
    target_tokens=64,
)
audio_enc.eval()

audio_features = {}
AUDIO_DIR = os.path.join(DATA_DIR, 'processed/audios')

for a in annotations:
    clip_id = a['clip_id']
    cache_path = os.path.join(FEAT_DIR, f'{clip_id}_audio.pt')
    if os.path.exists(cache_path):
        continue   # skip se já existe

    audio_path = os.path.join(AUDIO_DIR, f'{clip_id}.wav')
    if os.path.exists(audio_path):
        try:
            feat = audio_enc.encode(Path(audio_path))  # (1, 64, 768)
            torch.save(feat, cache_path)
            print(f'  Audio feat: {clip_id} {tuple(feat.shape)}')
        except Exception as e:
            # Zero tensor se falhar
            torch.save(torch.zeros(1, 64, 768), cache_path)
            print(f'  ⚠️  {clip_id}: {e}')
    else:
        torch.save(torch.zeros(1, 64, 768), cache_path)

del audio_enc
gc.collect(); torch.cuda.empty_cache()
print('\n✅ Audio features cached')

In [ ]:
# ============================================================
# CELL 7 — Context ViT + Text XLM-R Features
# ============================================================
from vie_gameemo.encoders.context_vit import ContextEncoder
from vie_gameemo.encoders.text_xlmr import XLMRTextEncoder

print('Loading Context ViT...')
ctx_enc = ContextEncoder(model_name='google/vit-base-patch16-224')
ctx_enc.eval()

FRAMES_BASE = os.path.join(DATA_DIR, 'processed/frames')
for a in annotations:
    clip_id = a['clip_id']
    cache_path = os.path.join(FEAT_DIR, f'{clip_id}_context.pt')
    if os.path.exists(cache_path):
        continue
    frames_dir = os.path.join(FRAMES_BASE, clip_id)
    if os.path.exists(frames_dir):
        frame_paths = sorted(Path(frames_dir).glob('*.jpg'))
        try:
            feat = ctx_enc.encode(frame_paths)  # (1, T, 768)
            torch.save(feat, cache_path)
        except Exception:
            torch.save(torch.zeros(1, 1, 768), cache_path)
    else:
        torch.save(torch.zeros(1, 1, 768), cache_path)

del ctx_enc
gc.collect(); torch.cuda.empty_cache()
print('✅ Context features cached')

# Text
print('Loading XLM-R text encoder...')
text_enc = XLMRTextEncoder(model_name='FacebookAI/xlm-roberta-base')
text_enc.eval()

for a in annotations:
    clip_id = a['clip_id']
    cache_path = os.path.join(FEAT_DIR, f'{clip_id}_text.pt')
    if os.path.exists(cache_path):
        continue
    transcript = a.get('transcript', '')
    try:
        feat = text_enc.encode(transcript)  # (1, T, 768)
        torch.save(feat, cache_path)
    except Exception:
        torch.save(torch.zeros(1, 1, 768), cache_path)

del text_enc
gc.collect(); torch.cuda.empty_cache()
print('✅ Text features cached')

In [ ]:
# ============================================================
# CELL 8 — Face ViT Features
# ============================================================
from vie_gameemo.encoders.face_vit import FaceEncoder

print('Loading Face ViT encoder...')
face_enc = FaceEncoder(model_name='trpakov/vit-face-expression')
face_enc.eval()

FACES_BASE = os.path.join(DATA_DIR, 'processed/faces')
FRAMES_BASE = os.path.join(DATA_DIR, 'processed/frames')

for a in annotations:
    clip_id = a['clip_id']
    cache_feat = os.path.join(FEAT_DIR, f'{clip_id}_face.pt')
    cache_flag = os.path.join(FEAT_DIR, f'{clip_id}_has_face.pt')
    if os.path.exists(cache_feat):
        continue

    # Thử faces crop trước, fallback sang frames gốc
    face_dir = os.path.join(FACES_BASE, clip_id)
    frames_dir = os.path.join(FRAMES_BASE, clip_id)
    search_dir = face_dir if os.path.exists(face_dir) and os.listdir(face_dir) else frames_dir

    if os.path.exists(search_dir):
        frame_paths = sorted(Path(search_dir).glob('*.jpg'))
        try:
            feat, has_face = face_enc.encode(frame_paths)  # (1, T, 768), bool
            torch.save(feat, cache_feat)
            torch.save(torch.tensor([has_face]), cache_flag)
        except Exception:
            torch.save(torch.zeros(1, 1, 768), cache_feat)
            torch.save(torch.tensor([False]), cache_flag)
    else:
        torch.save(torch.zeros(1, 1, 768), cache_feat)
        torch.save(torch.tensor([False]), cache_flag)

del face_enc
gc.collect(); torch.cuda.empty_cache()
print('✅ Face features cached')

# Kiểm tra
n_cached = len(list(Path(FEAT_DIR).glob('*_audio.pt')))
print(f'\nFeature cache: {n_cached}/{len(annotations)} clips')

## Bước 2 — Stage 1: Perception Training

In [ ]:
# ============================================================
# CELL 9 — Xây dựng DataLoader
# ============================================================
from torch.utils.data import DataLoader
from vie_gameemo.data.dataset import VieGameEmoDataset, collate_fn, make_splits

# Schema gaming_9 — phải khớp với EmotionLabel enum trong src/vie_gameemo/data/schemas.py
LABEL_NAMES = ['neutral', 'focus', 'hype', 'amused', 'tilted', 'sad', 'shocked', 'fear', 'disgusted']
LABEL2IDX = {l: i for i, l in enumerate(LABEL_NAMES)}

SPLITS_PATH = Path(ANNOT_DIR) / 'splits.json'

# Tạo splits JSON nếu chưa có
if not SPLITS_PATH.exists():
    make_splits(
        annotations_dir=Path(ANNOT_DIR),
        split_ratios=(0.70, 0.15, 0.10, 0.05),
        seed=42,
        output_path=SPLITS_PATH,
    )
    print(f'Splits → {SPLITS_PATH}')

def make_loader(split, shuffle=False):
    ds = VieGameEmoDataset(
        annotations_dir=Path(ANNOT_DIR),
        features_dir=Path(FEAT_DIR),
        split=split,
        splits_path=SPLITS_PATH,
        label2idx=LABEL2IDX,
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=True, collate_fn=collate_fn)

train_loader = make_loader('train', shuffle=True)
val_loader   = make_loader('val', shuffle=False)
test_loader  = make_loader('test_id', shuffle=False)

print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}')

In [ ]:
# ============================================================
# CELL 10 — Stage 1: Perception Training
# ============================================================
from types import SimpleNamespace
from vie_gameemo.utils.seed import set_seed
from vie_gameemo.training.perception import train_perception

set_seed(42)

# Build config namespace
cfg = SimpleNamespace(
    seed=42,
    paths=SimpleNamespace(
        checkpoints=CKPT_DIR,
        features=FEAT_DIR,
        annotations=ANNOT_DIR,
    ),
    fusion=SimpleNamespace(
        type=FUSION_TYPE,
        d_model=768,
        n_modalities=4,
        n_conv_blocks=4,
        kernel_size=3,
        align_to='audio',
        return_attention=False,
    ),
    classifier=SimpleNamespace(
        hidden_dim=256,
        n_classes=9,
        dropout=0.3,
        loss=SimpleNamespace(focal=SimpleNamespace(gamma=2.0, alpha=1.0)),
    ),
    training=SimpleNamespace(
        perception=SimpleNamespace(
            epochs=EPOCHS_PERC,
            batch_size=BATCH_SIZE,
            gradient_accumulation=GRAD_ACCUM,
            learning_rate=SimpleNamespace(fusion=LR_FUSION, classifier=LR_FUSION, encoders=0.0),
            weight_decay=0.01,
            warmup_ratio=0.1,
            grad_clip=1.0,
            mixed_precision=MIXED_PREC,
            early_stopping=SimpleNamespace(patience=EARLY_STOP_PAT),
            num_workers=2,
        ),
    ),
)

print(f'Training Perception: {EPOCHS_PERC} epochs | fusion={FUSION_TYPE} | device={device}')
best_ckpt = train_perception(
    cfg=cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
)
print(f'\n✅ Best checkpoint: {best_ckpt}')

In [ ]:
# ============================================================
# CELL 11 — Eval trên test_id
# ============================================================
from vie_gameemo.classifiers.mlp import EmotionClassifier
from vie_gameemo.fusion import get_fusion
from vie_gameemo.training.perception import evaluate, load_checkpoint
from vie_gameemo.evaluation.metrics import compute_metrics, format_confusion_matrix

fusion = get_fusion(
    FUSION_TYPE,
    d_model=768, n_modalities=4, n_conv_blocks=4,
    kernel_size=3, align_to='audio', return_attention=False,
).to(device)
classifier = EmotionClassifier(768, 256, 9, 0.3).to(device)
load_checkpoint(best_ckpt, fusion, classifier)

metrics = evaluate(fusion, classifier, test_loader, device, n_classes=9)
print('\n=== Test ID Results ===')
print(f'  Accuracy   : {metrics["accuracy"]:.4f}')
print(f'  Macro F1   : {metrics["macro_f1"]:.4f}')
print(f'  Weighted F1: {metrics["weighted_f1"]:.4f}')
print(f'  UAR        : {metrics["uar"]:.4f}')

# Per-class F1
all_preds, all_labels = [], []
fusion.eval(); classifier.eval()
with torch.no_grad():
    for batch in test_loader:
        a = batch['audio'].to(device)
        f = batch['face'].to(device)
        c = batch['context'].to(device)
        t = batch['text'].to(device)
        lab = batch['label'].to(device)
        hf = batch.get('has_face')
        if hf is not None: hf = hf.to(device)
        fused = fusion(a, f, c, t, has_face=hf)
        if isinstance(fused, tuple): fused = fused[0]
        preds = classifier(fused).argmax(-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(lab.cpu().tolist())

full_metrics = compute_metrics(all_labels, all_preds, n_classes=9, label_names=LABEL_NAMES)
print('\nPer-class F1:')
for label, f1 in full_metrics['per_class_f1'].items():
    bar = '█' * int(f1 * 20)
    print(f'  {label:<15}: {bar} {f1:.3f}')

print('\nConfusion matrix:')
print(format_confusion_matrix(full_metrics['confusion_matrix'], LABEL_NAMES))

## Bước 3 — Stage 2: Cognition Training (tùy chọn)

In [ ]:
# ============================================================
# CELL 12 — Cognition (Joint LLM + Adapter)
# ============================================================
if TRAIN_COGNITION:
    from vie_gameemo.training.cognition import train_cognition

    cog_cfg = SimpleNamespace(
        paths=cfg.paths,
        fusion=cfg.fusion,
        classifier=cfg.classifier,
        training=SimpleNamespace(
            cognition=SimpleNamespace(
                epochs=EPOCHS_COG,
                batch_size=4,
                gradient_accumulation=8,
                learning_rate=SimpleNamespace(llm=2e-5, llm_adapter=2e-4),
                loss_weights=SimpleNamespace(classification=1.0, reasoning_lm=0.5),
                lora=SimpleNamespace(
                    enabled=True, rank=16, alpha=32,
                    target_modules=['q_proj', 'v_proj'],
                ),
            ),
            perception=SimpleNamespace(weight_decay=0.01, warmup_ratio=0.1),
        ),
        llm=SimpleNamespace(
            base_model=SimpleNamespace(
                name=LLM_MODEL,
                quantization='4bit',
            )
        ),
    )

    cog_train_loader = make_loader('train', shuffle=True)
    cog_val_loader   = make_loader('val', shuffle=False)

    print(f'Training Cognition: {EPOCHS_COG} epochs | LLM={LLM_MODEL}')
    cog_ckpt = train_cognition(
        cfg=cog_cfg,
        perception_checkpoint=best_ckpt,
        train_loader=cog_train_loader,
        val_loader=cog_val_loader,
        device=device,
    )
    print(f'\n✅ Cognition checkpoint: {cog_ckpt}')
else:
    print('TRAIN_COGNITION=False — bỏ qua Stage 2')
    print('Để bật: đặt TRAIN_COGNITION = True ở CELL 2')

In [ ]:
# ============================================================
# CELL 13 — RLVR (LLM-4, tùy chọn)
# ============================================================
if TRAIN_RLVR:
    import subprocess
    print('Running RLVR cold start...')
    r = subprocess.run([
        'python', os.path.join(PROJECT_DIR, 'scripts/train_rlvr.py'),
        '--config', CONFIG_PATH,
        '--phase', 'cold-start',
        '--base-model', 'Qwen/Qwen2.5-1.5B-Instruct',  # nhỏ hơn cho Kaggle
        '--epochs', '2',
        '--annotations-dir', ANNOT_DIR,
    ], text=True)
    if r.returncode == 0:
        cold_start_dir = os.path.join(CKPT_DIR, 'llm4_coldstart')
        print(f'Cold start → {cold_start_dir}')
        print('Running RLVR GRPO...')
        subprocess.run([
            'python', os.path.join(PROJECT_DIR, 'scripts/train_rlvr.py'),
            '--config', CONFIG_PATH,
            '--phase', 'rlvr',
            '--resume-from', cold_start_dir,
            '--epochs', '1',
        ], text=True)
    else:
        print('⚠️  Cold start thất bại')
else:
    print('TRAIN_RLVR=False — bỏ qua RLVR')

In [ ]:
# ============================================================
# CELL 14 — Lưu checkpoint để download
# ============================================================
import zipfile, shutil

# List tất cả checkpoints
ckpt_files = list(Path(CKPT_DIR).glob('*.pt'))
print(f'Checkpoints ({len(ckpt_files)}):')
for f in ckpt_files:
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

# Archive
archive = os.path.join(WORKING, 'vie_gameemo_checkpoints.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ckpt in ckpt_files:
        zf.write(str(ckpt), f'checkpoints/{ckpt.name}')
    # Thêm config
    zf.write(CONFIG_PATH, 'config.yaml')

print(f'\n✅ Archive: {archive} ({os.path.getsize(archive)/1e6:.1f} MB)')
print('📥 Download: File browser → vie_gameemo_checkpoints.zip → chuột phải → Download')